# Transformer-based Speech Emotion Recognition (from scratch)

A small Transformer encoder trained from scratch on the same log-mel
spectrogram segments as the CNN / CNN+LSTM baselines.

- Input: `(3, 64, 64)` segments (log-mel + delta + delta²)
- Each of the 64 time frames is treated as a token (192-d = 64 mel × 3 channels)
- Evaluation: LOSO 10-fold, utterance-level accuracy via majority voting

In [1]:
import os
import sys
import math
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from ser_utils import (
    NUM_CLASSES,
    EMOTION_ENG_MAP,
    set_all_seeds,
    aggregate_segment_predictions,
    evaluate_and_report,
)

set_all_seeds(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## Configuration

In [2]:
# ── Paths ──────────────────────────────────────────────────────────
DATASET_DIR     = "../emodb_norm_loso"   # output of ser_preprocess.py
MODEL_DIR       = "models"
RESULTS_DIR     = "results"
SPLIT_MODE      = "loso"
NUM_FOLDS       = 10

# ── Transformer hyper-parameters ──────────────────────────────────
D_MODEL         = 128
N_HEADS         = 4
N_LAYERS        = 3
DIM_FFN         = 256
DROPOUT         = 0.1

# ── Training ──────────────────────────────────────────────────────
BATCH_SIZE      = 32
MAX_EPOCHS      = 50
LR              = 1e-3
WEIGHT_DECAY    = 0.01
PATIENCE        = 7
MAX_GRAD_NORM   = 1.0

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

## Dataset

In [3]:
class MelTransformerDataset(Dataset):
    """
    Loads preprocessed .npy segment data.
    Applies per-channel min-max scaling to [0, 1].
    """

    def __init__(self, fold_dir: str, split: str):
        self.X = np.load(os.path.join(fold_dir, f"X_{split}.npy"))           # (N, 3, 64, 64)
        self.y = np.load(os.path.join(fold_dir, f"y_{split}.npy"))            # (N,)
        self.utt_ids = np.load(os.path.join(fold_dir, f"utt_ids_{split}.npy"))  # (N,)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].astype(np.float32)          # (3, 64, 64)
        # Per-channel min-max scaling
        for c in range(x.shape[0]):
            cmin, cmax = x[c].min(), x[c].max()
            if cmax - cmin > 1e-8:
                x[c] = (x[c] - cmin) / (cmax - cmin)
            else:
                x[c] = 0.0
        return (
            torch.from_numpy(x),
            torch.tensor(self.y[idx], dtype=torch.long),
            self.utt_ids[idx],
        )


def build_dataloaders(fold_dir: str, batch_size: int = BATCH_SIZE):
    train_ds = MelTransformerDataset(fold_dir, "train")
    val_ds   = MelTransformerDataset(fold_dir, "validation")
    test_ds  = MelTransformerDataset(fold_dir, "test")
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, drop_last=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, val_loader, test_loader

## Model: MelTransformer

In [4]:
class MelTransformer(nn.Module):
    """
    Small Transformer encoder for segment-level SER.

    Input  : (B, 3, 64, 64)  – 3-channel mel spectrogram segment
    Tokens : each of the 64 time frames → 192-d (64 mels × 3 channels)
    Output : (B, num_classes) logits
    """

    def __init__(
        self,
        d_model: int = D_MODEL,
        n_heads: int = N_HEADS,
        n_layers: int = N_LAYERS,
        dim_ffn: int = DIM_FFN,
        dropout: float = DROPOUT,
        num_classes: int = NUM_CLASSES,
        seq_len: int = 64,
        input_dim: int = 192,     # 64 mels × 3 channels
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_ffn,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, 64, 64)
        B = x.size(0)
        # Reshape: each time frame = 1 token of dim 192
        x = x.permute(0, 3, 1, 2)           # (B, 64, 3, 64)
        x = x.reshape(B, 64, -1)            # (B, 64, 192)

        x = self.input_proj(x)               # (B, 64, d_model)
        x = x + self.pos_embed               # add positional encoding
        x = self.transformer(x)              # (B, 64, d_model)

        # Mean pooling over time
        x = x.mean(dim=1)                    # (B, d_model)

        x = self.norm(x)
        x = self.drop(x)
        return self.head(x)                  # (B, num_classes)


# Quick parameter count check
_m = MelTransformer()
print(f"MelTransformer parameters: {sum(p.numel() for p in _m.parameters()):,}")
del _m

MelTransformer parameters: 431,495


## Training

In [5]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_utt_ids = [], [], []
    for x, y, uids in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        preds = logits.argmax(1)
        total_loss += loss.item() * x.size(0)
        correct += (preds == y).sum().item()
        total += x.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        all_utt_ids.extend(uids)
    return (
        total_loss / total,
        correct / total,
        np.array(all_preds),
        np.array(all_labels),
        np.array(all_utt_ids),
    )


def train_one_fold(fold_dir: str, fold_idx: int):
    """Train on one LOSO fold; return (seg_preds, seg_true, utt_ids) on test set."""
    print(f"\n{'─' * 50}")
    print(f"  Fold {fold_idx:02d}")
    print(f"{'─' * 50}")

    train_loader, val_loader, test_loader = build_dataloaders(fold_dir)
    model = MelTransformer().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_val_loss = float("inf")
    best_state = None
    wait = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if epoch % 10 == 0 or wait == 0 or epoch == 1:
            print(
                f"  Epoch {epoch:3d} | "
                f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
                f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} | "
                f"wait={wait}"
            )

        if wait >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

    # Save best model
    ckpt_path = os.path.join(MODEL_DIR, f"transformer_fold{fold_idx:02d}.pth")
    torch.save(best_state, ckpt_path)
    print(f"  Saved → {ckpt_path}")

    # Evaluate on test set with best model
    model.load_state_dict(best_state)
    _, test_acc, seg_preds, seg_true, utt_ids = evaluate(model, test_loader, criterion)
    print(f"  Test segment acc: {test_acc:.3f}")

    return seg_preds, seg_true, utt_ids

## LOSO Training Loop

In [6]:
all_seg_preds = []
all_seg_true  = []
all_utt_ids   = []

for fold_idx in range(1, NUM_FOLDS + 1):
    fold_dir = os.path.join(DATASET_DIR, f"fold{fold_idx:02d}")
    seg_preds, seg_true, utt_ids = train_one_fold(fold_dir, fold_idx)
    all_seg_preds.append(seg_preds)
    all_seg_true.append(seg_true)
    all_utt_ids.append(utt_ids)

all_seg_preds = np.concatenate(all_seg_preds)
all_seg_true  = np.concatenate(all_seg_true)
all_utt_ids   = np.concatenate(all_utt_ids)

print(f"\nTotal test segments: {len(all_seg_preds)}")


──────────────────────────────────────────────────
  Fold 01
──────────────────────────────────────────────────
  Epoch   1 | train_loss=1.8673 train_acc=0.247 | val_loss=2.2018 val_acc=0.209 | wait=0
  Epoch   2 | train_loss=1.3823 train_acc=0.422 | val_loss=1.3498 val_acc=0.526 | wait=0
  Epoch   4 | train_loss=1.2185 train_acc=0.503 | val_loss=1.0239 val_acc=0.568 | wait=0
  Epoch  10 | train_loss=0.9656 train_acc=0.610 | val_loss=1.1534 val_acc=0.574 | wait=6
  Early stopping at epoch 11
  Saved → models\transformer_fold01.pth
  Test segment acc: 0.438

──────────────────────────────────────────────────
  Fold 02
──────────────────────────────────────────────────
  Epoch   1 | train_loss=1.9878 train_acc=0.195 | val_loss=1.8942 val_acc=0.269 | wait=0
  Epoch   2 | train_loss=1.6413 train_acc=0.350 | val_loss=1.3295 val_acc=0.440 | wait=0
  Epoch   4 | train_loss=1.3001 train_acc=0.474 | val_loss=1.0838 val_acc=0.601 | wait=0
  Epoch  10 | train_loss=0.9543 train_acc=0.620 | val_lo

## Evaluation — Utterance-Level (Majority Voting)

In [7]:
y_true_utt, y_pred_utt = aggregate_segment_predictions(
    all_seg_preds, all_seg_true, all_utt_ids
)

acc, report, cm = evaluate_and_report(
    y_true_utt, y_pred_utt,
    model_name="MelTransformer",
    results_dir=RESULTS_DIR,
    split_mode=SPLIT_MODE,
)


  MelTransformer  —  LOSO
  Utterance-level accuracy: 57.20%
              precision    recall  f1-score   support

       Anger       0.63      0.89      0.74       127
     Boredom       0.51      0.54      0.53        81
        Fear       0.53      0.23      0.32        69
   Happiness       0.44      0.23      0.30        71
     Sadness       0.77      0.95      0.85        62
     Disgust       0.38      0.59      0.46        46
     Neutral       0.55      0.39      0.46        79

    accuracy                           0.57       535
   macro avg       0.55      0.55      0.52       535
weighted avg       0.56      0.57      0.54       535

  Confusion matrix saved → results\cm_MelTransformer_loso.png


## Model: Enhanced Conv-Attentive Transformer with SpecAugment and training tricks

In [8]:
from sklearn.metrics import accuracy_score
from ser_utils import compute_class_weights


ENHANCED_MODEL_NAME = "MelTransformerEnhanced"
ENHANCED_BATCH_SIZE = 32
ENHANCED_MAX_EPOCHS = 60
ENHANCED_PATIENCE = 10
ENHANCED_LR = 7e-4
ENHANCED_WEIGHT_DECAY = 0.02
ENHANCED_LABEL_SMOOTHING = 0.05
ENHANCED_WARMUP_RATIO = 0.1
ENHANCED_RUN_FULL = False


class FoldStandardizedMelDataset(Dataset):
    def __init__(self, fold_dir: str, split: str, mean=None, std=None, train_mode: bool = False):
        self.X = np.load(os.path.join(fold_dir, f"X_{split}.npy")).astype(np.float32)
        self.y = np.load(os.path.join(fold_dir, f"y_{split}.npy")).astype(np.int64)
        self.utt_ids = np.load(os.path.join(fold_dir, f"utt_ids_{split}.npy"))
        self.train_mode = train_mode
        if mean is None or std is None:
            self.mean = self.X.mean(axis=(0, 2, 3), keepdims=True)
            self.std = self.X.std(axis=(0, 2, 3), keepdims=True) + 1e-6
        else:
            self.mean = mean
            self.std = std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = (self.X[idx:idx + 1] - self.mean) / self.std
        x = x.squeeze(0)
        if self.train_mode:
            x = self.spec_augment(x)
        return torch.from_numpy(x), torch.tensor(self.y[idx], dtype=torch.long), self.utt_ids[idx]

    @staticmethod
    def spec_augment(x: np.ndarray, max_freq_mask: int = 8, max_time_mask: int = 10):
        x = x.copy()
        if np.random.rand() < 0.6:
            f = np.random.randint(0, max_freq_mask + 1)
            if f > 0:
                f0 = np.random.randint(0, max(1, x.shape[1] - f + 1))
                x[:, f0:f0 + f, :] = 0
        if np.random.rand() < 0.6:
            t = np.random.randint(0, max_time_mask + 1)
            if t > 0:
                t0 = np.random.randint(0, max(1, x.shape[2] - t + 1))
                x[:, :, t0:t0 + t] = 0
        return x


def build_enhanced_dataloaders(fold_dir: str):
    train_ds = FoldStandardizedMelDataset(fold_dir, "train", train_mode=True)
    val_ds = FoldStandardizedMelDataset(fold_dir, "validation", mean=train_ds.mean, std=train_ds.std)
    test_ds = FoldStandardizedMelDataset(fold_dir, "test", mean=train_ds.mean, std=train_ds.std)
    return (
        DataLoader(train_ds, batch_size=ENHANCED_BATCH_SIZE, shuffle=True, drop_last=False),
        DataLoader(val_ds, batch_size=ENHANCED_BATCH_SIZE, shuffle=False, drop_last=False),
        DataLoader(test_ds, batch_size=ENHANCED_BATCH_SIZE, shuffle=False, drop_last=False),
        train_ds,
    )


class ConvAttentiveMelTransformer(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, d_model: int = 192, n_heads: int = 6, n_layers: int = 4):
        super().__init__()
        self.frontend = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 48, kernel_size=3, padding=1),
            nn.BatchNorm2d(48),
            nn.GELU(),
            nn.Dropout2d(0.1),
        )
        self.freq_pool = nn.AdaptiveAvgPool2d((16, 64))
        self.input_proj = nn.Linear(48 * 16, d_model)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.randn(1, 65, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=0.2,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        z = self.frontend(x)
        z = self.freq_pool(z)
        z = z.permute(0, 3, 1, 2).reshape(x.size(0), 64, -1)
        z = self.input_proj(z)
        cls = self.cls.expand(x.size(0), -1, -1)
        z = torch.cat([cls, z], dim=1) + self.pos
        z = self.encoder(z)
        z = self.norm(z[:, 0])
        return self.head(z)


def enhanced_segment_eval(model, loader, criterion):
    model.eval()
    total_loss, total, correct = 0.0, 0, 0
    preds, labels, utt_ids = [], [], []
    with torch.no_grad():
        for x, y, uids in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            pred = logits.argmax(dim=1)
            total_loss += loss.item() * x.size(0)
            correct += (pred == y).sum().item()
            total += x.size(0)
            preds.extend(pred.cpu().numpy())
            labels.extend(y.cpu().numpy())
            utt_ids.extend(uids)
    y_utt, p_utt = aggregate_segment_predictions(np.array(preds), np.array(labels), np.array(utt_ids))
    return {
        "loss": total_loss / max(1, total),
        "seg_acc": correct / max(1, total),
        "utt_acc": accuracy_score(y_utt, p_utt),
        "preds": np.array(preds),
        "labels": np.array(labels),
        "utt_ids": np.array(utt_ids),
    }


def train_enhanced_transformer_fold(fold_idx: int):
    fold_dir = os.path.join(DATASET_DIR, f"fold{fold_idx:02d}")
    train_loader, val_loader, test_loader, train_ds = build_enhanced_dataloaders(fold_dir)
    weights = compute_class_weights(train_ds.y).to(DEVICE)
    model = ConvAttentiveMelTransformer().to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=ENHANCED_LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), lr=ENHANCED_LR, weight_decay=ENHANCED_WEIGHT_DECAY)

    steps = max(1, len(train_loader) * ENHANCED_MAX_EPOCHS)
    warmup = int(steps * ENHANCED_WARMUP_RATIO)

    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        progress = (step - warmup) / max(1, steps - warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    best_score, best_state, wait = -1.0, None, 0
    global_step = 0

    for epoch in range(1, ENHANCED_MAX_EPOCHS + 1):
        model.train()
        total_loss, total, correct = 0.0, 0, 0
        for x, y, _ in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            global_step += 1
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += x.size(0)

        val = enhanced_segment_eval(model, val_loader, criterion)
        if val["utt_acc"] > best_score:
            best_score = val["utt_acc"]
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
        print(
            f"Fold {fold_idx:02d} epoch {epoch:02d} | "
            f"train_loss={total_loss / max(1, total):.4f} train_acc={correct / max(1, total):.3f} | "
            f"val_utt_acc={val['utt_acc']:.3f} val_seg_acc={val['seg_acc']:.3f} | wait={wait}"
        )
        if wait >= ENHANCED_PATIENCE:
            break

    model.load_state_dict(best_state)
    ckpt = os.path.join(MODEL_DIR, f"mel_transformer_enhanced_fold{fold_idx:02d}.pth")
    torch.save(best_state, ckpt)
    test = enhanced_segment_eval(model, test_loader, criterion)
    print(f"Fold {fold_idx:02d} enhanced test utterance acc: {test['utt_acc']:.3f}")
    return test["preds"], test["labels"], test["utt_ids"]


if ENHANCED_RUN_FULL:
    enhanced_preds, enhanced_labels, enhanced_uids = [], [], []
    for fold_idx in range(1, NUM_FOLDS + 1):
        p, y, u = train_enhanced_transformer_fold(fold_idx)
        enhanced_preds.append(p)
        enhanced_labels.append(y)
        enhanced_uids.append(u)
    enhanced_preds = np.concatenate(enhanced_preds)
    enhanced_labels = np.concatenate(enhanced_labels)
    enhanced_uids = np.concatenate(enhanced_uids)
    enhanced_y_true, enhanced_y_pred = aggregate_segment_predictions(enhanced_preds, enhanced_labels, enhanced_uids)
    evaluate_and_report(
        enhanced_y_true,
        enhanced_y_pred,
        model_name=ENHANCED_MODEL_NAME,
        results_dir=RESULTS_DIR,
        split_mode=SPLIT_MODE,
    )


## Training

In [9]:
RUN_ENHANCED_4A_EXPERIMENT = True
# Use None for all 10 LOSO folds, or a list such as [1, 2, 3].
ENHANCED_4A_FOLDS_TO_RUN = None
# Output model name.
ENHANCED_4A_MODEL_NAME = "MelTransformerEnhanced"

if RUN_ENHANCED_4A_EXPERIMENT:
    set_all_seeds(42)
    _folds = ENHANCED_4A_FOLDS_TO_RUN or list(range(1, NUM_FOLDS + 1))
    enhanced_4a_seg_preds = []
    enhanced_4a_seg_true = []
    enhanced_4a_utt_ids = []

    for _fold_idx in _folds:
        _preds, _labels, _uids = train_enhanced_transformer_fold(_fold_idx)
        enhanced_4a_seg_preds.append(_preds)
        enhanced_4a_seg_true.append(_labels)
        enhanced_4a_utt_ids.append(_uids)

    enhanced_4a_seg_preds = np.concatenate(enhanced_4a_seg_preds)
    enhanced_4a_seg_true = np.concatenate(enhanced_4a_seg_true)
    enhanced_4a_utt_ids = np.concatenate(enhanced_4a_utt_ids)
    print(f"Enhanced 4a trained/evaluated folds: {_folds}")
    print(f"Enhanced 4a total test segments: {len(enhanced_4a_seg_preds)}")
else:
    print("Set RUN_ENHANCED_4A_EXPERIMENT = True in the config cell above to run enhanced 4a training.")


D:\Temp\ipykernel_31872\373421440.py:92: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


Fold 01 epoch 01 | train_loss=1.7434 train_acc=0.315 | val_utt_acc=0.379 val_seg_acc=0.546 | wait=0
Fold 01 epoch 02 | train_loss=1.2517 train_acc=0.547 | val_utt_acc=0.638 val_seg_acc=0.665 | wait=0
Fold 01 epoch 03 | train_loss=1.1085 train_acc=0.615 | val_utt_acc=0.483 val_seg_acc=0.558 | wait=1
Fold 01 epoch 04 | train_loss=1.0413 train_acc=0.657 | val_utt_acc=0.586 val_seg_acc=0.633 | wait=2
Fold 01 epoch 05 | train_loss=1.0024 train_acc=0.678 | val_utt_acc=0.483 val_seg_acc=0.548 | wait=3
Fold 01 epoch 06 | train_loss=0.9337 train_acc=0.706 | val_utt_acc=0.569 val_seg_acc=0.645 | wait=4
Fold 01 epoch 07 | train_loss=0.9202 train_acc=0.725 | val_utt_acc=0.690 val_seg_acc=0.701 | wait=0
Fold 01 epoch 08 | train_loss=0.8495 train_acc=0.751 | val_utt_acc=0.759 val_seg_acc=0.707 | wait=0
Fold 01 epoch 09 | train_loss=0.7739 train_acc=0.793 | val_utt_acc=0.707 val_seg_acc=0.673 | wait=1
Fold 01 epoch 10 | train_loss=0.7717 train_acc=0.794 | val_utt_acc=0.569 val_seg_acc=0.639 | wait=2


D:\Temp\ipykernel_31872\373421440.py:92: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


Fold 02 epoch 01 | train_loss=1.7643 train_acc=0.298 | val_utt_acc=0.628 val_seg_acc=0.576 | wait=0
Fold 02 epoch 02 | train_loss=1.2633 train_acc=0.560 | val_utt_acc=0.558 val_seg_acc=0.579 | wait=1
Fold 02 epoch 03 | train_loss=1.0973 train_acc=0.629 | val_utt_acc=0.698 val_seg_acc=0.625 | wait=0
Fold 02 epoch 04 | train_loss=1.0228 train_acc=0.674 | val_utt_acc=0.744 val_seg_acc=0.675 | wait=0
Fold 02 epoch 05 | train_loss=0.9449 train_acc=0.708 | val_utt_acc=0.581 val_seg_acc=0.548 | wait=1
Fold 02 epoch 06 | train_loss=0.9338 train_acc=0.721 | val_utt_acc=0.628 val_seg_acc=0.604 | wait=2
Fold 02 epoch 07 | train_loss=0.8914 train_acc=0.733 | val_utt_acc=0.628 val_seg_acc=0.647 | wait=3
Fold 02 epoch 08 | train_loss=0.8635 train_acc=0.748 | val_utt_acc=0.651 val_seg_acc=0.669 | wait=4
Fold 02 epoch 09 | train_loss=0.7912 train_acc=0.787 | val_utt_acc=0.674 val_seg_acc=0.616 | wait=5
Fold 02 epoch 10 | train_loss=0.7566 train_acc=0.810 | val_utt_acc=0.791 val_seg_acc=0.690 | wait=0


D:\Temp\ipykernel_31872\373421440.py:92: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


...

Fold 08 epoch 01 | train_loss=1.7714 train_acc=0.307 | val_utt_acc=0.429 val_seg_acc=0.407 | wait=0
Fold 08 epoch 02 | train_loss=1.3364 train_acc=0.532 | val_utt_acc=0.589 val_seg_acc=0.508 | wait=0
Fold 08 epoch 03 | train_loss=1.1474 train_acc=0.616 | val_utt_acc=0.482 val_seg_acc=0.543 | wait=1
Fold 08 epoch 04 | train_loss=1.0909 train_acc=0.642 | val_utt_acc=0.661 val_seg_acc=0.609 | wait=0
Fold 08 epoch 05 | train_loss=1.0034 train_acc=0.697 | val_utt_acc=0.589 val_seg_acc=0.593 | wait=1
Fold 08 epoch 06 | train_loss=0.9859 train_acc=0.697 | val_utt_acc=0.464 val_seg_acc=0.503 | wait=2
Fold 08 epoch 07 | train_loss=0.9701 train_acc=0.715 | val_utt_acc=0.679 val_seg_acc=0.622 | wait=0
Fold 08 epoch 08 | train_loss=0.8410 train_acc=0.774 | val_utt_acc=0.804 val_seg_acc=0.654 | wait=0
Fold 08 epoch 09 | train_loss=0.8317 train_acc=0.775 | val_utt_acc=0.679 val_seg_acc=0.604 | wait=1
Fold 08 epoch 10 | train_loss=0.7726 train_acc=0.804 | val_utt_acc=0.679 val_seg_acc=0.657 | wait=2


D:\Temp\ipykernel_31872\373421440.py:92: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


Fold 09 epoch 01 | train_loss=1.8178 train_acc=0.285 | val_utt_acc=0.437 val_seg_acc=0.457 | wait=0
Fold 09 epoch 02 | train_loss=1.3227 train_acc=0.553 | val_utt_acc=0.521 val_seg_acc=0.513 | wait=0
Fold 09 epoch 03 | train_loss=1.1664 train_acc=0.609 | val_utt_acc=0.535 val_seg_acc=0.512 | wait=0
Fold 09 epoch 04 | train_loss=1.0807 train_acc=0.656 | val_utt_acc=0.775 val_seg_acc=0.651 | wait=0
Fold 09 epoch 05 | train_loss=1.0167 train_acc=0.681 | val_utt_acc=0.620 val_seg_acc=0.560 | wait=1
Fold 09 epoch 06 | train_loss=0.9775 train_acc=0.701 | val_utt_acc=0.732 val_seg_acc=0.669 | wait=2
Fold 09 epoch 07 | train_loss=0.9291 train_acc=0.731 | val_utt_acc=0.789 val_seg_acc=0.664 | wait=0
Fold 09 epoch 08 | train_loss=0.8884 train_acc=0.746 | val_utt_acc=0.732 val_seg_acc=0.656 | wait=1
Fold 09 epoch 09 | train_loss=0.8340 train_acc=0.768 | val_utt_acc=0.789 val_seg_acc=0.656 | wait=2
Fold 09 epoch 10 | train_loss=0.7966 train_acc=0.794 | val_utt_acc=0.761 val_seg_acc=0.694 | wait=3


D:\Temp\ipykernel_31872\373421440.py:92: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


Fold 10 epoch 01 | train_loss=1.7844 train_acc=0.310 | val_utt_acc=0.653 val_seg_acc=0.528 | wait=0
Fold 10 epoch 02 | train_loss=1.3061 train_acc=0.559 | val_utt_acc=0.714 val_seg_acc=0.607 | wait=0
Fold 10 epoch 03 | train_loss=1.0840 train_acc=0.657 | val_utt_acc=0.612 val_seg_acc=0.551 | wait=1
Fold 10 epoch 04 | train_loss=1.0273 train_acc=0.676 | val_utt_acc=0.714 val_seg_acc=0.650 | wait=2
Fold 10 epoch 05 | train_loss=1.0006 train_acc=0.705 | val_utt_acc=0.735 val_seg_acc=0.672 | wait=0
Fold 10 epoch 06 | train_loss=0.9544 train_acc=0.719 | val_utt_acc=0.673 val_seg_acc=0.636 | wait=1
Fold 10 epoch 07 | train_loss=0.9290 train_acc=0.726 | val_utt_acc=0.694 val_seg_acc=0.624 | wait=2
Fold 10 epoch 08 | train_loss=0.8804 train_acc=0.758 | val_utt_acc=0.633 val_seg_acc=0.621 | wait=3
Fold 10 epoch 09 | train_loss=0.8337 train_acc=0.775 | val_utt_acc=0.776 val_seg_acc=0.678 | wait=0
Fold 10 epoch 10 | train_loss=0.7907 train_acc=0.786 | val_utt_acc=0.776 val_seg_acc=0.698 | wait=1


## Evaluation

In [10]:
enhanced_4a_y_true, enhanced_4a_y_pred = aggregate_segment_predictions(
    enhanced_4a_seg_preds,
    enhanced_4a_seg_true,
    enhanced_4a_utt_ids,
)
_full_run = ENHANCED_4A_FOLDS_TO_RUN is None or len(ENHANCED_4A_FOLDS_TO_RUN) == NUM_FOLDS
_split_name = SPLIT_MODE if _full_run else f"{SPLIT_MODE}_partial"
enhanced_4a_acc, enhanced_4a_report, enhanced_4a_cm = evaluate_and_report(
    enhanced_4a_y_true,
    enhanced_4a_y_pred,
    model_name=ENHANCED_4A_MODEL_NAME,
    results_dir=RESULTS_DIR,
    split_mode=_split_name,
)


  MelTransformerEnhanced  —  LOSO
  Utterance-level accuracy: 75.33%
              precision    recall  f1-score   support

       Anger       0.84      0.90      0.87       127
     Boredom       0.73      0.65      0.69        81
        Fear       0.63      0.58      0.61        69
   Happiness       0.63      0.68      0.65        71
     Sadness       0.86      0.89      0.87        62
     Disgust       0.90      0.78      0.84        46
     Neutral       0.69      0.72      0.70        79

    accuracy                           0.75       535
   macro avg       0.75      0.74      0.75       535
weighted avg       0.75      0.75      0.75       535

  Confusion matrix saved → results\cm_MelTransformerEnhanced_loso.png
